# Analytic nonlinear feature obfuscation with two MLP blocks
Consider a residual stream MLP architecture, like the one shown below:

![](model_architecture.svg)

The model is trying to learn `y = sat(x, -c, c)` (the saturation function, equivalently `y = max(-c, min(c, x))`). 
 `x` is a vector of dense features sampled uniformly from `[-3, 3]`, and `c` is a scalar sampled uniformly in `[1, 2]`. This is a fairly easy function to learn - it just takes `2*len(x)` neurons to learn exactly if you use ReLUs. But the harder question is, can a model do this without storing `c` in the residual stream, in a way that can be linearly probed?

This is an important question - although this is a toy model, its structure is reminiscent of how transformer models work, just on a much smaller scale. Normally, LLMs learn to store "features" as directions in activation space, such that you can train a linear probe to detect them. This is really useful! We can use this as a cheap monitor to detect things like whether the model is lying or whether the user's prompt is dangerous. You might think that if we can detect deception, we can just add the probe's score as a loss term in training, to align the model. However, training on mech-interp monitors like probes is considered [a forbidden technique](https://www.lesswrong.com/posts/mpmsK8KKysgSKDm2T/the-most-forbidden-technique), because the model could very well just learn to obfuscate its representations instead of learning what you wanted it to learn - and now you have a misaligned model *and* one less way to monitor it.

In this post I'll show that the model can at least theoretically represent `c` (itself meant to represent a feature we care about) in a non-linear fashion, in a way invisible to difference-of-means probes on every other layer, by means of an explicit construction (credit where credit's due: Claude came up with this encoding). Before we get started, though, I want to highlight some features of this architecture:
- The learned function is not linearly separable in `c` - so once the model has finished computing `y_i` for one `x_i` feature, it can just leave `y_i` in the residual stream. This is vaguely analogous to how probe accuracy seems to get a bit worse as you check the last layers of a transformer model, as the model starts shifting its representation from more abstract concepts to the exact text it wants to output. 
- Because it takes at least `2 * len(x)` neurons to implement `y`, as long as the width of the MLP blocks is less than `2 * len(x)`, we can be sure the model has represented `c` somehow in the first layer (or the model hasn't actually learned the task fully). More generally, the model needs to encounter at least `2 * len(x)` neurons before it can "forget" about `c`. In practical experiments, we can use this property to guarentee that the model is representing `c` at a particular early layer. 
- For simplicity, we'll also assume that the residual stream is arbitrarily wide, and that the embedding and unembedding matrices are rectangular identity matrices (i.e. ones on the main diagonal, zero elsewhere).


In [76]:

import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML

relu = lambda z: np.maximum(z, 0.0)
C_LO, C_HI = 1.0, 2.0
X_LO, X_HI = -3.0, 3.0


def animate_slider_html(param_values, build_frame, prefix, layout=None, height=450):
    """build_frame(p) -> (list_of_go_traces, dict_of_frame_layout_overrides_or_None)."""

    def frame_for(p):
        traces, layout_overrides = build_frame(p)
        return go.Frame(data=traces, name=f"{p:.3f}", layout=go.Layout(**(layout_overrides or {})))

    p0 = param_values[len(param_values) // 3]
    traces0, layout0 = build_frame(p0)
    fig = go.Figure(data=traces0, frames=[frame_for(p) for p in param_values])
    if layout:
        fig.update_layout(**layout)
    if layout0:
        fig.update_layout(**layout0)
    fig.update_layout(
        autosize=True,
        height=height,
        margin=dict(l=50, r=20, t=60, b=50),
        sliders=[{
            "currentvalue": {"prefix": prefix},
            "steps": [
                {
                    "method": "animate",
                    "label": f"{p:.2f}",
                    "args": [[f"{p:.3f}"], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}}],
                }
                for p in param_values
            ],
        }],
    )
    return HTML(fig.to_html(full_html=False, include_plotlyjs="cdn", config={"responsive": True}))


## Encoding: `v1(x1, c)` and `v2(x1, c)`
We'll encode `c` in the first MLP block, using two channels and borrowing an unrelated feature $x_1$:

```
v1(x1, c) = -2*ReLU(-x1 - c)   + 2*ReLU(x1 + c   - 3) - c + 1.5
v2(x1, c) = -4*ReLU(-x1 - c/2) + 4*ReLU(x1 + c/2 - 3) - c + 3.0
```
These have the property that $\int_{-3}^3 v_1 dx_1 = \int_{-3}^3 v_2 dx_1 = 0$. In other words, their mean value is always 0, regardless of $c$. This makes them invisible to difference-of-means probes. The next plot visualizes these functions. Note in particular the locations of the kinks:

- `v1`: `x1 = -c` and `x1 = 3-c`
- `v2`: `x1 = -c/2` and `x1 = 3-c/2`

Over the valid range of $1 \leq c \leq 2$, these four kink positions are *always* in the same left-to-right order: `-c < -c/2 < 3-c < 3-c/2`. This reliably carves `x1` into 5 bands.

As a bit of bookkeeping - we'll also erase `c` from the residual stream here, using an always-on neuron (e.g. add `-ReLU(c + 100) - 100)` to the channel contianing `c`).


In [77]:
import sympy as sp

x_expr, c_expr = sp.symbols('x1 c', real=True)
relu_expr = lambda z: sp.Max(z, 0)

v1_expr = -2 * relu_expr(-x_expr - c_expr) + 2 * relu_expr(x_expr - 3 + c_expr) - c_expr + sp.Rational(3, 2)
v2_expr = -4 * relu_expr(-x_expr - c_expr/2) + 4 * relu_expr(x_expr + c_expr/2 - 3) - c_expr + 3

I1 = sp.simplify(sp.integrate(v1_expr, (x_expr, -3, 3)))
I2 = sp.simplify(sp.integrate(v2_expr, (x_expr, -3, 3)))

# As of writing - Sympy's inequality handling capabilities aren't great. Work around it by repeatedly substituting these known identities (since 1 <= c <= 2)
subs_facts = {
    sp.Max(-3, -c_expr): -c_expr,
    sp.Max(-3, 3 - c_expr): 3-c_expr,
    sp.Min(3, -c_expr): -c_expr,
    sp.Min(3, 3 - c_expr): 3-c_expr,
    sp.Max(-3, -c_expr/2): -c_expr/2,
    sp.Min(3, -c_expr/2): -c_expr/2,
    sp.Min(3, 3 - c_expr/ 2):3 - c_expr/ 2,
    sp.Max(-3, 3 - c_expr/ 2):3 - c_expr/ 2
}
def refine(expr):
    return sp.simplify(expr.subs(subs_facts))
print(refine(refine(I1))) # identically 0
print(refine(refine(I2)))


0
0


In [ ]:
def v1f(x, c):
    return -2 * relu(-x - c) + 2 * relu(x - 3 + c) - c + 1.5


def v2f(x, c):
    return -4 * relu(-x - c / 2) + 4 * relu(x + c / 2 - 3) - c + 3.0


x_grid = np.linspace(X_LO, X_HI, 121)
c_values = np.round(np.arange(C_LO, C_HI + 1e-9, 0.04), 2)


def step1_frame(c):
    v1, v2 = v1f(x_grid, c), v2f(x_grid, c)
    traces = [
        go.Scatter(x=x_grid, y=v1, name="v1(x1, c)", line=dict(color="#1f77b4", width=2)),
        go.Scatter(x=x_grid, y=v2, name="v2(x1, c)", line=dict(color="#ff7f0e", width=2)),
    ]
    shapes = [
        dict(type="line", x0=x, x1=x, y0=-6, y1=8, line=dict(color="#1f77b4", dash="dot", width=1))
        for x in (-c, 3 - c)
    ] + [
        dict(type="line", x0=x, x1=x, y0=-6, y1=8, line=dict(color="#ff7f0e", dash="dot", width=1))
        for x in (-c / 2, 3 - c / 2)
    ]
    return traces, {"shapes": shapes}


animate_slider_html(
    c_values,
    step1_frame,
    prefix="c = ",
    layout={
        "xaxis": {"title": "x1", "range": [X_LO, X_HI]},
        "yaxis": {"range": [-6, 8]},
        "title": "v1(x1) and v2(x1)  (dotted lines: kink positions)",
    },
)


## Decoding 

If you already knew which of the 5 bands `x1` was in, reading `c` back out
would be trivial, since each each segment of `v1` and `v2` is linear and can be inverted to recover `c`. In fact, you'd only need one of the `v` channels. For example, if you knew `x1 < -c`, then `v1` simplfies to `1.5-c` and you could recover `c = 1.5-v1`. 

Unfortunately, we can't predict ahead of time which band `x1` is going to be in, because we don't even know where the bands are (remember, we've erased `c`, which defines the kink locations). Instead, 

Fortunately (or unfortunately for AI safety people?), there's a workaround:  

Each of these 5 formulas, evaluated **everywhere** (not just its own
window), is a different piecewise-linear curve in `x1`. Only inside its own
window is it flat and equal to `c` — outside, it just tracks whatever
`v1`/`v2` are actually doing there. Slide `c` to see all 5 curves move, and
slide `x1` to see which one is "in charge" at that point (the vertical grey
line) — its value should land right on the black diamond marker at height
`c`.

The catch: to actually use this, the network would need to *select* which
of the 5 formulas applies based on `x1` — an if/else (a "min-tree"). That
selection can't be written as `affine + Σ relu(affine)`; it needs more than
one hidden layer. That's exactly the piece Step 3 replaces.


In [3]:

windows = [
    (-3.0, -2.0, "x1<=-2:  v1-2x1-1.5", lambda x, v1, v2: v1 - 2 * x - 1.5),
    (-2.0, -1.0, "[-2,-1]: v2-4x1-3", lambda x, v1, v2: v2 - 4 * x - 3),
    (-1.0, 1.0, "[-1,1]:  1.5-v1", lambda x, v1, v2: 1.5 - v1),
    (1.0, 2.0, "[1,2]:   3-v2", lambda x, v1, v2: 3 - v2),
    (2.0, 3.0, "x1>=2:   v1-2x1+4.5", lambda x, v1, v2: v1 - 2 * x + 4.5),
]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
x1_values = np.round(np.arange(X_LO, X_HI + 1e-9, 0.25), 2)


def step2_curve_ys(c):
    v1, v2 = v1f(x_grid, c), v2f(x_grid, c)
    return [f(x_grid, v1, v2) for (lo, hi, label, f) in windows]


def step2_curve_traces(c):
    return [
        go.Scatter(x=x_grid, y=y, name=label, line=dict(color=col, width=2))
        for y, (lo, hi, label, f), col in zip(step2_curve_ys(c), windows, colors)
    ]


def marker_trace(c, x1):
    return go.Scatter(
        x=[x1], y=[c], mode="markers",
        marker=dict(color="black", size=10, symbol="diamond"),
        name="target (x1, c)",
    )


_c0, _x10 = c_values[len(c_values) // 3], 0.0
_curve_trace_indices = [0, 1, 2, 3, 4]
_marker_trace_index = 5
_all_trace_indices = _curve_trace_indices + [_marker_trace_index]

fig2 = go.Figure(data=step2_curve_traces(_c0) + [marker_trace(_c0, _x10)])
fig2.update_layout(
    xaxis=dict(title="x1", range=[X_LO, X_HI]),
    yaxis=dict(range=[-4, 7], title="candidate decode value"),
    title="Step 2: five candidate decode formulas -- each flat & correct only in its own window",
    autosize=True,
    height=620,
    margin=dict(l=50, r=20, t=60, b=190),
    shapes=[dict(type="line", x0=_x10, x1=_x10, y0=-4, y1=7, line=dict(color="grey", width=1))],
    sliders=[
        {
            "y": -0.12,
            "currentvalue": {"prefix": "c = "},
            "steps": [
                {
                    "method": "update",
                    "label": f"{c:.2f}",
                    # Only "y" is touched (x_grid never changes with c, and
                    # the marker's own x is owned by the x1 slider) -- so
                    # this can never disturb whatever x1 is currently set to.
                    "args": [{"y": step2_curve_ys(c) + [[c]]}, {}, _all_trace_indices],
                }
                for c in c_values
            ],
        },
        {
            "y": -0.4,
            "currentvalue": {"prefix": "x1 = "},
            "steps": [
                {
                    "method": "update",
                    "label": f"{x1:.2f}",
                    # Only the marker's "x" is touched here -- curves' x_grid
                    # and every trace's "y" are left alone, so this can never
                    # disturb whatever c is currently set to.
                    "args": [
                        {"x": [[x1]]},
                        {"shapes[0].x0": x1, "shapes[0].x1": x1},
                        [_marker_trace_index],
                    ],
                }
                for x1 in x1_values
            ],
        },
    ],
)
# Two fully independent sliders, both stacked below the plot, implemented
# with plain "update" steps (no frames/animate at all): the c slider only
# ever touches the "y" arrays of all 6 traces; the x1 slider only ever
# touches the "x" array of the marker trace. Since each slider only writes
# the one attribute it owns, dragging either one can never clobber whatever
# the other slider last set -- everything needed for both is baked into the
# page as precomputed per-step data, no Python involved after page load.
HTML(fig2.to_html(full_html=False, include_plotlyjs="cdn", config={"responsive": True}))


## Step 3 — curve-vanishing atoms: a "which side" test with no gating

Instead of gating on `x1` directly, build **affine functions of
`(x1, v1, v2)`** — not of `c`! — that are identically zero along one entire
kink curve, for *every* `c`. E.g. for the curve `x1 = -c/2`:

```
P(x1, v1, v2) = -2*x1 + v1 - 1.5
```

Check it: at `x1 = -c/2`, `v1` is still on its flat plateau (`1.5 - c`), so
`P = c + (1.5 - c) - 1.5 = 0`, for *any* `c`. `P` doesn't need to know `c`
— it's already implied by where `v1`/`v2` are.

The useful part is `P`'s **sign** away from the curve: it tells you which
side of the (`c`-dependent, otherwise unknown) kink `x1` is on. Two of the
four kink curves (`x1=-c/2` and `x1=3-c/2`) admit affine combinations like
this that stay a *consistent* sign on each side, for every `c` simultaneously
(found below by a brute-force angle scan over the 2-parameter family of
curve-vanishing affines — the other two curves, `x1=-c` and `x1=3-c`, turn
out to be "creases" with no valid one-sided version, so they're not used).

**Aside — are these atoms monotonic in `x1`?** Not necessarily! The plots
below print a monotonic flag per atom. The two "pure" atoms (angle exactly
0°, i.e. just the raw curve-vanishing basis vector) are simple monotonic
ramps. But most of the 8 chosen atoms are optimized *combinations* of two
basis directions, and those can wobble in the middle — they were only
selected because they cross zero exactly **once**, staying one-signed
everywhere except at their own curve, not because they're monotonic.


In [82]:

# same procedure as period2_net.py / period2_decode.py, at a coarser (but
# still exact-to-1e-13) grid so this cell runs in under a second.
bases = {2: [(-2, 1, 0, -1.5), (-2, 0, 1, -3)], 4: [(0, 1, 0, -1.5), (-2, 0, 1, 3)]}
xs_s = np.linspace(X_LO, X_HI, 401)
cs_s = np.linspace(C_LO, C_HI, 81)
Xg, Cg = np.meshgrid(xs_s, cs_s, indexing="ij")
V1g, V2g = v1f(Xg, Cg), v2f(Xg, Cg)
curve_pos = {2: -Cg / 2, 4: 3 - Cg / 2}


def combo(j, s1, s2):
    b1, b2 = bases[j]
    return tuple(s1 * u + s2 * v for u, v in zip(b1, b2))


def find_two_valid(j, side):
    got = []
    for ang in np.linspace(0, 2 * np.pi, 720, endpoint=False):
        s1, s2 = np.cos(ang), np.sin(ang)
        co = combo(j, s1, s2)
        P = co[0] * Xg + co[1] * V1g + co[2] * V2g + co[3]
        right, left = Xg > curve_pos[j] + 1e-6, Xg < curve_pos[j] - 1e-6
        pr, pl = (P[right] > 1e-9).mean(), (P[left] > 1e-9).mean()
        ok = (pr > 1 - 1e-9 and pl < 1e-9) if side == "R" else (pl > 1 - 1e-9 and pr < 1e-9)
        if ok:
            got.append((s1, s2))
    M = np.array(got)
    dots = M @ M[0]
    i1 = int(np.argmin(np.abs(dots)))
    return [tuple(M[0]), tuple(M[i1])]


atom_coeffs, atom_labels = [], []
for j in (2, 4):
    for side in ("L", "R"):
        for s1, s2 in find_two_valid(j, side):
            atom_coeffs.append(combo(j, s1, s2))
            atom_labels.append(f"curve x1={'-c/2' if j == 2 else '3-c/2'}, side {side}")

print(f"found {len(atom_coeffs)} sign-valid atoms:")
for lbl, co in zip(atom_labels, atom_coeffs):
    print(f"  {lbl:22s}  (a,b,g,d) = {tuple(round(v, 3) for v in co)}")


found 8 sign-valid atoms:
  curve x1=-c/2, side L   (a,b,g,d) = (np.float64(-2.0), np.float64(1.0), np.float64(0.0), np.float64(-1.5))
  curve x1=-c/2, side L   (a,b,g,d) = (np.float64(-0.564), np.float64(0.834), np.float64(-0.552), np.float64(0.405))
  curve x1=-c/2, side R   (a,b,g,d) = (np.float64(0.564), np.float64(-0.834), np.float64(0.552), np.float64(-0.405))
  curve x1=-c/2, side R   (a,b,g,d) = (np.float64(2.345), np.float64(-0.982), np.float64(-0.191), np.float64(2.045))
  curve x1=3-c/2, side L  (a,b,g,d) = (np.float64(-1.414), np.float64(-0.707), np.float64(0.707), np.float64(3.182))
  curve x1=3-c/2, side L  (a,b,g,d) = (np.float64(0.313), np.float64(-0.988), np.float64(-0.156), np.float64(1.012))
  curve x1=3-c/2, side R  (a,b,g,d) = (np.float64(0.0), np.float64(1.0), np.float64(0.0), np.float64(-1.5))
  curve x1=3-c/2, side R  (a,b,g,d) = (np.float64(1.402), np.float64(0.713), np.float64(-0.701), np.float64(-3.173))


In [81]:

from plotly.subplots import make_subplots

_c_mid = c_values[len(c_values) // 2]


def _atom_label_with_stats(lbl, a, b, g, d):
    P = a * x_grid + b * v1f(x_grid, _c_mid) + g * v2f(x_grid, _c_mid) + d
    signs = np.sign(P)
    nz = signs[signs != 0]
    n_sign_changes = int((np.diff(nz) != 0).sum())
    monotonic = bool(np.all(np.diff(P) >= -1e-9) or np.all(np.diff(P) <= 1e-9))
    # These properties are invariant across c by construction (see Step 3 text
    # above), so it's safe to compute them once rather than per-frame.
    return f"{lbl}<br>sign changes={n_sign_changes}, monotonic={monotonic}"


_subplot_titles = [_atom_label_with_stats(lbl, *co) for lbl, co in zip(atom_labels, atom_coeffs)]
# 3x3 grid (8 atoms + 1 empty cell) -- narrow enough for a blog column, without
# getting as tall as a 4x2 grid would.
fig3 = make_subplots(rows=3, cols=3, subplot_titles=_subplot_titles, vertical_spacing=0.1, horizontal_spacing=0.08)

_c0 = c_values[len(c_values) // 3]
for i, (a, b, g, d) in enumerate(atom_coeffs):
    row, col = divmod(i, 3)
    P0 = a * x_grid + b * v1f(x_grid, _c0) + g * v2f(x_grid, _c0) + d
    fig3.add_trace(
        go.Scatter(x=x_grid, y=P0, line=dict(color="#9467bd", width=2), showlegend=False),
        row=row + 1, col=col + 1,
    )

_frames3 = []
for c_expr in c_values:
    v1, v2 = v1f(x_grid, c_expr), v2f(x_grid, c_expr)
    fdata = [go.Scatter(y=a * x_grid + b * v1 + g * v2 + d) for a, b, g, d in atom_coeffs]
    _frames3.append(go.Frame(data=fdata, name=f"{c_expr:.3f}"))
fig3.frames = _frames3

fig3.update_layout(
    title="the 8 curve-vanishing atoms P_j(x1)",
    autosize=True,
    height=900,
    margin=dict(l=50, r=20, t=80, b=50),
    sliders=[{
        "currentvalue": {"prefix": "c = "},
        "steps": [
            {
                "method": "animate",
                "label": f"{c:.2f}",
                "args": [[f"{c:.3f}"], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}}],
            }
            for c in c_values
        ],
    }],
)
HTML(fig3.to_html(full_html=False, include_plotlyjs="cdn", config={"responsive": True}))


NameError: name 'atom_labels' is not defined

## Step 4 — summing ReLUs of the atoms to build `c`

Once the 8 atoms are one-sided, `c` is just the standard "piecewise-linear
function as base + Σ ReLU ramps" identity:

```
c = a*x1 + b*v1 + g*v2 + k + Σ_j  w_j * relu(P_j(x1, v1, v2))
```

The base affine coefficients `(a,b,g,k)` and the 8 weights `w_j` are found
by least-squares over the `(x1,c)` grid (this is exactly solvable — the
residual is ~`1e-14`, i.e. exact up to floating point).

In the plot: **dashed** lines show each term `w_j * P_j` *without* the ReLU
clip (i.e. what it would be if it kept going negative); **solid** lines show
what's actually used, `w_j * relu(P_j)` (zero on one side, linear on the
other). The thick black line is the running sum, which should sit flat on
the grey reference line at `c`.


In [6]:

featg = [np.ones_like(Xg), Xg, V1g, V2g] + [
    relu(a * Xg + b * V1g + g * V2g + d) for (a, b, g, d) in atom_coeffs
]
A_mat = np.stack([f.ravel() for f in featg], axis=1)
w_dec, *_ = np.linalg.lstsq(A_mat, Cg.ravel(), rcond=None)
max_err = np.abs(A_mat @ w_dec - Cg.ravel()).max()
print(f"decode fit: c = base(x1,v1,v2) + sum w_j*relu(atom_j), max|error| = {max_err:.2e}")
print("base (const, x1, v1, v2):", np.round(w_dec[:4], 4))
print("atom weights w_j:", np.round(w_dec[4:], 4))


decode fit: c = base(x1,v1,v2) + sum w_j*relu(atom_j), max|error| = 9.10e-15
base (const, x1, v1, v2): [-0.3281 -0.7519  0.0184  0.1592]
atom weights w_j: [ 2.0982 -1.5768 -1.7954  0.7272  0.119   0.7265  1.1686  0.0144]


In [7]:

_palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f"]


def step4_frame(c):
    v1, v2 = v1f(x_grid, c), v2f(x_grid, c)
    base = w_dec[0] + w_dec[1] * x_grid + w_dec[2] * v1 + w_dec[3] * v2
    traces = [go.Scatter(x=x_grid, y=base, line=dict(color="black", width=1, dash="dash"), name="base affine")]
    running = base.copy()
    for j, ((a, b, g, d), w) in enumerate(zip(atom_coeffs, w_dec[4:])):
        Pj = a * x_grid + b * v1 + g * v2 + d
        unclipped, clipped = w * Pj, w * relu(Pj)
        col = _palette[j % len(_palette)]
        traces.append(go.Scatter(x=x_grid, y=unclipped, line=dict(color=col, width=1, dash="dot"), showlegend=False))
        traces.append(go.Scatter(x=x_grid, y=clipped, line=dict(color=col, width=1.5), showlegend=False))
        running = running + clipped
    traces.append(go.Scatter(x=x_grid, y=running, line=dict(color="black", width=3), name="running sum (decoded c)"))
    return traces, {"shapes": [dict(type="line", x0=X_LO, x1=X_HI, y0=c, y1=c, line=dict(color="grey", width=1))]}


animate_slider_html(
    c_values,
    step4_frame,
    prefix="c = ",
    layout={
        "xaxis": {"title": "x1", "range": [X_LO, X_HI]},
        "yaxis": {"range": [-2, 5]},
        "title": "Step 4: base affine + sum of w_j * relu(atom_j)  =  decoded c",
    },
)


## Step 5 — wiring it into the network (bookkeeping, no plot needed)

Given the formula from Step 4, the network schedule (`period2_net.py`) is:

- **Block `2k`** (an unprobed layer): compute the 8
  `Q_j = relu(atom_j(x1, v1, v2))` and write them to 8 fresh residual
  dimensions. Everything on the right-hand side (`x1`, `v1`, `v2`) is
  already resident in the residual stream, so this is one ordinary MLP
  block (linear-in, ReLU, linear-out).
- **Block `2k+1`** (a probed layer): fold `c_hat = affine(x1, v1, v2,
  Q_1..Q_8)` directly into the *pre-activations* of the coordinate-finishing
  neurons — `relu(x_i - c_hat)` and `relu(-x_i - c_hat)` — rather than
  materializing `c_hat` as its own residual dimension. Since `c_hat` is a
  linear function of dimensions already in the residual, this needs no
  extra ReLU: it's a "linear read" that happens for free inside the next
  block's input projection. The same block also clears the 8 `Q_j` dims
  back to zero (one always-on neuron per dim) so the next probed layer sees
  no leftover trace of the decode — only `[finished coords, pending x, 0,
  v1, v2, 0]`, which is exactly the mean-constant content the probe sees at
  every probed layer.
